In [1]:
import glob
import pickle
import numpy as np
import pandas as pd

OUT_DIR = "./output2.3"

result_paths = sorted(glob.glob(f"{OUT_DIR}/sim2_final_result*.pkl"))
if not result_paths:
    raise FileNotFoundError(f"No sim2_final result files found in {OUT_DIR}")

data = []
for path in result_paths:
    with open(path, "rb") as f:
        chunk = pickle.load(f)
    data.extend(chunk if isinstance(chunk, list) else [chunk])

thetas     = [13.0, 13.1, 13.2, 13.3, 13.4]
theta_keys = [f"{t:g}" for t in thetas]

# AIS config used in simulation_scenario2_final.py
AIS_N_PATHS         = 300
AIS_N_LEVELS        = 20
AIS_MOVES_PER_LEVEL = 5

print(f"Loaded {len(data)} replications, thetas={thetas}")


Loaded 50 replications, thetas=[13.0, 13.1, 13.2, 13.3, 13.4]


In [3]:
theta_keys[0]="13.0"

In [4]:
def l1(a, b):
    return float(np.sum(np.abs(np.asarray(a) - np.asarray(b))))

def ci_iou(lo_m, hi_m, lo_ref, hi_ref):
    intersection = np.maximum(0, np.minimum(hi_m, hi_ref) - np.maximum(lo_m, lo_ref))
    ref_len = hi_ref - lo_ref
    return float(np.nanmean(np.where(ref_len > 0, intersection / ref_len, np.nan)))

def ess(normw):
    w = np.asarray(normw, dtype=float)
    return float(1.0 / np.sum(w ** 2))

def normw_from_logw(logw):
    lw = np.asarray(logw, float)
    w  = np.exp(lw - np.max(lw))
    return w / w.sum()

rows = []
for x in data:
    mcmc_m  = np.asarray(x["MCMC"])
    mcmc_lo = x["MCMC_quantiles"]["0.025"]
    mcmc_hi = x["MCMC_quantiles"]["0.975"]

    for key in theta_keys:
        # --- RPS ---
        rps_nw = normw_from_logw(x[f"RPS_{key}_logw"])
        rows.append(dict(
            method="RPS",
            theta=float(key),
            n_states=x[f"RPS_{key}_n_states"],
            ESS=ess(rps_nw),
            ESS_ratio=ess(rps_nw) / x[f"RPS_{key}_n_states"],
            L1_mean=l1(x[f"RPS_{key}"], mcmc_m),
            L1_q025=l1(x[f"RPS_{key}_quantiles"]["0.025"], mcmc_lo),
            L1_q975=l1(x[f"RPS_{key}_quantiles"]["0.975"], mcmc_hi),
            IoU=ci_iou(x[f"RPS_{key}_quantiles"]["0.025"],
                       x[f"RPS_{key}_quantiles"]["0.975"], mcmc_lo, mcmc_hi),
            runtime_s=x.get(f"RPS_{key}_time", np.nan),
        ))

        # --- AIS ---
        ais_out = x[f"AIS_{key}_output"]
        e_ais   = ess(ais_out.normw)
        rows.append(dict(
            method="AIS",
            theta=float(key),
            n_states=len(ais_out.terminals),
            ESS=e_ais,
            ESS_ratio=e_ais / len(ais_out.terminals),
            L1_mean=l1(x[f"AIS_{key}"], mcmc_m),
            L1_q025=l1(x[f"AIS_{key}_quantiles"]["0.025"], mcmc_lo),
            L1_q975=l1(x[f"AIS_{key}_quantiles"]["0.975"], mcmc_hi),
            IoU=ci_iou(x[f"AIS_{key}_quantiles"]["0.025"],
                       x[f"AIS_{key}_quantiles"]["0.975"], mcmc_lo, mcmc_hi),
            runtime_s=x.get(f"AIS_{key}_time", np.nan),
        ))

        # --- PB ---
        pb_nw = normw_from_logw(x[f"PB_{key}_logw"])
        rows.append(dict(
            method="PB",
            theta=float(key),
            n_states=x[f"PB_{key}_n_states"],
            ESS=ess(pb_nw),
            ESS_ratio=ess(pb_nw) / x[f"PB_{key}_n_states"],
            L1_mean=l1(x[f"PB_{key}"], mcmc_m),
            L1_q025=l1(x[f"PB_{key}_quantiles"]["0.025"], mcmc_lo),
            L1_q975=l1(x[f"PB_{key}_quantiles"]["0.975"], mcmc_hi),
            IoU=ci_iou(x[f"PB_{key}_quantiles"]["0.025"],
                       x[f"PB_{key}_quantiles"]["0.975"], mcmc_lo, mcmc_hi),
            runtime_s=x.get(f"PB_{key}_time", np.nan),
        ))

df = pd.DataFrame(rows)

summary = (
    df.groupby("method")
    [["n_states", "L1_mean", "L1_q025", "L1_q975", "IoU", "ESS", "ESS_ratio", "runtime_s"]]
    .mean()
    .round(4)
)
summary


,n_states,L1_mean,L1_q025,L1_q975,IoU,ESS,ESS_ratio,runtime_s
method,,,,,,,,
AIS,300.000,1.4999,0.0721,0.0760,0.9981,201.5653,0.6719,142.1053
PB,400.752,10.8668,1.2063,1.6974,0.9545,400.7520,1.0000,619.0357
RPS,100.752,14.9624,5.8240,9.0676,0.8193,100.7520,1.0000,NaN


In [5]:
# Same table broken out by theta
summary_theta = (
    df.groupby(["theta", "method"])
    [["n_states", "L1_mean", "L1_q025", "L1_q975", "IoU", "ESS", "ESS_ratio", "runtime_s"]]
    .mean()
    .round(4)
)
summary_theta


n_states  L1_mean  L1_q025  L1_q975     IoU       ESS  \
theta method                                                          
13.0  AIS       300.00   1.3694   0.0685   0.0708  0.9982  215.9756   
      PB        341.88  11.1774   1.3059   1.4569  0.9559  341.8800   
      RPS        41.88  15.2591   6.8591  11.3082  0.7829   41.8800   
13.1  AIS       300.00   1.4219   0.0693   0.0711  0.9982  211.0521   
      PB        374.86  10.7622   1.3308   1.4770  0.9554  374.8600   
      RPS        74.86  14.4900   5.5808   8.5229  0.8255   74.8600   
13.2  AIS       300.00   1.4881   0.0716   0.0777  0.9979  190.3232   
      PB        402.30  11.0402   1.0261   1.6509  0.9574  402.3000   
      RPS       102.30  15.3977   5.5747   8.5094  0.8275  102.3000   
13.3  AIS       300.00   1.6334   0.0766   0.0829  0.9980  193.4369   
      PB        434.60  11.0067   1.2619   1.9302  0.9518  434.6000   
      RPS       134.60  15.5048   5.5683   8.5083  0.8297  134.6000   
13.4  AIS       300.00   1.5869   0.0745   0.0777  0.9980  197.0387   
      PB        450.12  10.3472   1.1069   1.9718  0.9523  450.1200   
      RPS       150.12  14.1604   5.5373   8.4893  0.8307  150.1200   

              ESS_ratio  runtime_s  
theta method                        
13.0  AIS        0.7199   141.5149  
      PB         1.0000   603.4623  
      RPS        1.0000        NaN  
13.1  AIS        0.7035   141.9710  
      PB         1.0000   620.3921  
      RPS        1.0000        NaN  
13.2  AIS        0.6344   142.0841  
      PB         1.0000   621.1175  
      RPS        1.0000        NaN  
13.3  AIS        0.6448   141.6532  
      PB         1.0000   622.6985  
      RPS        1.0000        NaN  
13.4  AIS        0.6568   143.3032  
      PB         1.0000   627.5081  
      RPS        1.0000        NaN